# Chapter 2.5: API Layer Design

## Learning Objectives

By the end of this notebook, you will:

1. **Understand OpenAI API compatibility** and why it matters for the LLM ecosystem
2. **Implement the Chat Completions API** endpoint with full parameter support
3. **Build SSE (Server-Sent Events) streaming** for real-time token delivery
4. **Handle request parameters** correctly: temperature, top_p, max_tokens, stop sequences
5. **Implement token usage tracking** and billing-ready reporting
6. **Design robust error handling** for production-grade serving
7. **Build a complete OpenAI-compatible API server** using FastAPI

## Prerequisites

- Chapter 2.1: Request Lifecycle
- Basic HTTP/REST API knowledge
- Python async/await concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part2/chapter_2.5_api_layer.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part2/chapter_2.5_api_layer.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why OpenAI API Compatibility Matters

The OpenAI API has become the **de facto standard** for LLM APIs. By implementing the same interface, serving systems like vLLM and SGLang enable:

- **Drop-in replacement**: Applications built for OpenAI can switch to self-hosted models without code changes
- **Ecosystem compatibility**: All tools, SDKs, and libraries that support OpenAI work automatically
- **Migration path**: Easy to move between cloud and on-premise deployment

### The Two Main API Endpoints

| Endpoint | Purpose | Input | Output |
|----------|---------|-------|--------|
| `/v1/chat/completions` | Chat-style generation | Messages list | Assistant response |
| `/v1/completions` | Raw text completion | Raw text prompt | Continued text |

Modern LLM serving almost exclusively uses the **Chat Completions API**, as it supports system prompts, multi-turn conversations, and structured outputs.

In [ ]:
import json
import time
import uuid
import asyncio
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any, Union, AsyncGenerator
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# === Data Models: OpenAI API Schema ===

@dataclass
class ChatMessage:
    """A single message in a chat conversation."""
    role: str  # "system", "user", "assistant", "tool"
    content: str
    name: Optional[str] = None
    
    def to_dict(self):
        d = {'role': self.role, 'content': self.content}
        if self.name:
            d['name'] = self.name
        return d

@dataclass
class ChatCompletionRequest:
    """OpenAI-compatible chat completion request."""
    model: str
    messages: List[ChatMessage]
    temperature: float = 1.0
    top_p: float = 1.0
    n: int = 1  # Number of completions to generate
    stream: bool = False
    stop: Optional[Union[str, List[str]]] = None
    max_tokens: Optional[int] = None
    presence_penalty: float = 0.0
    frequency_penalty: float = 0.0
    logprobs: Optional[bool] = None
    top_logprobs: Optional[int] = None
    user: Optional[str] = None
    seed: Optional[int] = None
    
    @classmethod
    def from_dict(cls, data: dict) -> 'ChatCompletionRequest':
        messages = [ChatMessage(**m) for m in data.get('messages', [])]
        return cls(
            model=data.get('model', ''),
            messages=messages,
            temperature=data.get('temperature', 1.0),
            top_p=data.get('top_p', 1.0),
            n=data.get('n', 1),
            stream=data.get('stream', False),
            stop=data.get('stop'),
            max_tokens=data.get('max_tokens'),
            presence_penalty=data.get('presence_penalty', 0.0),
            frequency_penalty=data.get('frequency_penalty', 0.0),
            user=data.get('user'),
            seed=data.get('seed'),
        )

@dataclass
class UsageInfo:
    """Token usage information for billing."""
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

@dataclass
class ChatChoice:
    """A single completion choice."""
    index: int
    message: ChatMessage
    finish_reason: Optional[str] = None  # "stop", "length", "content_filter"

@dataclass
class ChatCompletionResponse:
    """OpenAI-compatible chat completion response."""
    id: str
    object: str = "chat.completion"
    created: int = 0
    model: str = ""
    choices: List[ChatChoice] = field(default_factory=list)
    usage: Optional[UsageInfo] = None
    
    def __post_init__(self):
        if self.created == 0:
            self.created = int(time.time())
    
    def to_dict(self):
        return {
            'id': self.id,
            'object': self.object,
            'created': self.created,
            'model': self.model,
            'choices': [{
                'index': c.index,
                'message': c.message.to_dict(),
                'finish_reason': c.finish_reason,
            } for c in self.choices],
            'usage': {
                'prompt_tokens': self.usage.prompt_tokens,
                'completion_tokens': self.usage.completion_tokens,
                'total_tokens': self.usage.total_tokens,
            } if self.usage else None,
        }

# Demo: Create a complete request-response cycle
request_data = {
    "model": "llama-3-8b-instruct",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    "temperature": 0.7,
    "max_tokens": 100,
    "stream": False
}

request = ChatCompletionRequest.from_dict(request_data)
print("=== Parsed Request ===")
print(f"Model: {request.model}")
print(f"Messages: {len(request.messages)}")
for m in request.messages:
    print(f"  [{m.role}]: {m.content[:50]}...")
print(f"Temperature: {request.temperature}")
print(f"Max tokens: {request.max_tokens}")
print(f"Stream: {request.stream}")

# Simulated response
response = ChatCompletionResponse(
    id=f"chatcmpl-{uuid.uuid4().hex[:12]}",
    model=request.model,
    choices=[ChatChoice(
        index=0,
        message=ChatMessage(role="assistant", content="The capital of France is Paris."),
        finish_reason="stop"
    )],
    usage=UsageInfo(prompt_tokens=25, completion_tokens=8, total_tokens=33)
)

print(f"\n=== Response ===")
print(json.dumps(response.to_dict(), indent=2))

## 2. Streaming with Server-Sent Events (SSE)

Streaming is critical for user experience: instead of waiting for the entire response, users see tokens appear in real-time.

### SSE Protocol

SSE is a simple HTTP-based protocol for server-to-client streaming:
- Response content type: `text/event-stream`
- Each event: `data: {json}\n\n`
- Final event: `data: [DONE]\n\n`

### Streaming Response Format

Each chunk uses `chat.completion.chunk` object type and sends a **delta** (not the full message):

```json
{"id": "chatcmpl-xxx", "object": "chat.completion.chunk",
 "choices": [{"index": 0, "delta": {"content": "The"}, "finish_reason": null}]}
```

In [ ]:
@dataclass
class StreamChunk:
    """A single streaming chunk."""
    id: str
    model: str
    delta_content: Optional[str] = None
    delta_role: Optional[str] = None
    finish_reason: Optional[str] = None
    
    def to_sse(self) -> str:
        """Convert to SSE format."""
        delta = {}
        if self.delta_role:
            delta['role'] = self.delta_role
        if self.delta_content is not None:
            delta['content'] = self.delta_content
        
        chunk = {
            'id': self.id,
            'object': 'chat.completion.chunk',
            'created': int(time.time()),
            'model': self.model,
            'choices': [{
                'index': 0,
                'delta': delta,
                'finish_reason': self.finish_reason,
            }]
        }
        return f"data: {json.dumps(chunk)}\n\n"


class StreamingResponseGenerator:
    """Generates SSE streaming responses."""
    
    def __init__(self, request_id: str, model: str):
        self.request_id = request_id
        self.model = model
        self.tokens_sent = 0
    
    def generate_role_chunk(self) -> str:
        """First chunk: sends the role."""
        chunk = StreamChunk(self.request_id, self.model, delta_role='assistant')
        return chunk.to_sse()
    
    def generate_content_chunk(self, text: str) -> str:
        """Content chunk: sends a text delta."""
        self.tokens_sent += 1
        chunk = StreamChunk(self.request_id, self.model, delta_content=text)
        return chunk.to_sse()
    
    def generate_final_chunk(self, finish_reason: str = 'stop') -> str:
        """Final chunk: signals completion."""
        chunk = StreamChunk(self.request_id, self.model, finish_reason=finish_reason)
        return chunk.to_sse()
    
    @staticmethod
    def generate_done() -> str:
        return "data: [DONE]\n\n"

# Simulate streaming response
print("=== Simulated SSE Stream ===")
print("(Each block is one SSE event)\n")

streamer = StreamingResponseGenerator("chatcmpl-demo123", "llama-3-8b")

# Role chunk
print(streamer.generate_role_chunk())

# Content chunks
words = ["The", " capital", " of", " France", " is", " Paris", "."]
for word in words:
    print(streamer.generate_content_chunk(word))

# Final chunk
print(streamer.generate_final_chunk('stop'))
print(streamer.generate_done())

## 3. Request Parameter Handling

The API must correctly handle all generation parameters. Each parameter affects how tokens are sampled.

### Critical Parameters

| Parameter | Range | Effect | Default |
|-----------|-------|--------|--------|
| `temperature` | [0, 2] | Scales logits before softmax. 0=greedy, higher=more random | 1.0 |
| `top_p` | [0, 1] | Nucleus sampling: consider tokens with cumulative prob <= p | 1.0 |
| `max_tokens` | [1, context_limit] | Maximum tokens to generate | Model-dependent |
| `stop` | string or list | Stop generation when these sequences appear | None |
| `presence_penalty` | [-2, 2] | Penalize tokens that have appeared (regardless of count) | 0 |
| `frequency_penalty` | [-2, 2] | Penalize tokens proportional to their frequency | 0 |
| `n` | [1, ...] | Number of independent completions to generate | 1 |
| `seed` | int | For reproducible generation | None |

In [ ]:
import numpy as np

class SamplingParams:
    """Encapsulates all sampling parameters with validation."""
    
    def __init__(self, request: ChatCompletionRequest):
        self.temperature = self._validate_range(request.temperature, 0, 2, 'temperature')
        self.top_p = self._validate_range(request.top_p, 0, 1, 'top_p')
        self.max_tokens = request.max_tokens or 4096
        self.presence_penalty = self._validate_range(request.presence_penalty, -2, 2, 'presence_penalty')
        self.frequency_penalty = self._validate_range(request.frequency_penalty, -2, 2, 'frequency_penalty')
        self.n = max(1, request.n)
        self.seed = request.seed
        
        # Normalize stop sequences
        if isinstance(request.stop, str):
            self.stop = [request.stop]
        elif isinstance(request.stop, list):
            self.stop = request.stop
        else:
            self.stop = []
    
    def _validate_range(self, value, min_val, max_val, name):
        if value < min_val or value > max_val:
            raise ValueError(f"{name} must be in [{min_val}, {max_val}], got {value}")
        return value
    
    def apply_penalties(self, logits: np.ndarray, 
                        token_counts: Dict[int, int]) -> np.ndarray:
        """Apply presence and frequency penalties to logits."""
        modified_logits = logits.copy()
        
        for token_id, count in token_counts.items():
            if count > 0:
                # Presence penalty: applied equally to all seen tokens
                modified_logits[token_id] -= self.presence_penalty
                # Frequency penalty: proportional to count
                modified_logits[token_id] -= self.frequency_penalty * count
        
        return modified_logits
    
    def sample(self, logits: np.ndarray) -> int:
        """Sample a token using configured parameters."""
        if self.temperature == 0:
            return int(np.argmax(logits))
        
        # Temperature scaling
        scaled = logits / self.temperature
        
        # Softmax
        exp_logits = np.exp(scaled - np.max(scaled))
        probs = exp_logits / exp_logits.sum()
        
        # Top-p filtering
        if self.top_p < 1.0:
            sorted_idx = np.argsort(probs)[::-1]
            cumsum = np.cumsum(probs[sorted_idx])
            mask = cumsum <= self.top_p
            mask[0] = True
            filtered = np.zeros_like(probs)
            filtered[sorted_idx[mask]] = probs[sorted_idx[mask]]
            probs = filtered / filtered.sum()
        
        return int(np.random.choice(len(probs), p=probs))

# Demo: Sampling with different parameters
np.random.seed(42)
vocab_size = 50
logits = np.random.randn(vocab_size) * 3

configs = [
    {'temperature': 0, 'top_p': 1.0},       # Greedy
    {'temperature': 0.7, 'top_p': 1.0},     # Standard
    {'temperature': 1.0, 'top_p': 0.9},     # Nucleus
    {'temperature': 1.5, 'top_p': 1.0},     # Creative
]

print("=== Sampling Parameter Effects ===")
for cfg in configs:
    req = ChatCompletionRequest.from_dict({
        'model': 'test', 'messages': [{'role': 'user', 'content': 'hi'}], **cfg
    })
    sp = SamplingParams(req)
    
    # Sample 100 times
    samples = [sp.sample(logits.copy()) for _ in range(100)]
    unique = len(set(samples))
    most_common = max(set(samples), key=samples.count)
    
    print(f"  temp={cfg['temperature']}, top_p={cfg['top_p']}: "
          f"{unique} unique tokens, most common=ID {most_common} ({samples.count(most_common)}%)")

## 4. Token Usage Tracking

Accurate token counting is essential for:
- **Billing**: Charging per token (input and output priced differently)
- **Rate limiting**: Enforcing tokens-per-minute limits
- **Monitoring**: Understanding resource consumption
- **Context window management**: Ensuring we don't exceed model limits

In [ ]:
class TokenUsageTracker:
    """Tracks token usage for billing and monitoring."""
    
    def __init__(self, input_price_per_1k: float = 0.15,
                 output_price_per_1k: float = 0.60):
        self.input_price = input_price_per_1k / 1000
        self.output_price = output_price_per_1k / 1000
        
        # Aggregated stats
        self.total_requests = 0
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.request_history: List[dict] = []
    
    def record_request(self, request_id: str, model: str,
                       prompt_tokens: int, completion_tokens: int,
                       latency_ms: float):
        """Record a completed request's usage."""
        self.total_requests += 1
        self.total_input_tokens += prompt_tokens
        self.total_output_tokens += completion_tokens
        
        cost = (prompt_tokens * self.input_price + 
                completion_tokens * self.output_price)
        
        record = {
            'request_id': request_id,
            'model': model,
            'prompt_tokens': prompt_tokens,
            'completion_tokens': completion_tokens,
            'total_tokens': prompt_tokens + completion_tokens,
            'cost_usd': cost,
            'latency_ms': latency_ms,
            'tokens_per_second': completion_tokens / (latency_ms / 1000) if latency_ms > 0 else 0,
            'timestamp': time.time(),
        }
        self.request_history.append(record)
        return record
    
    def get_usage(self, request_id: str) -> UsageInfo:
        for r in self.request_history:
            if r['request_id'] == request_id:
                return UsageInfo(
                    prompt_tokens=r['prompt_tokens'],
                    completion_tokens=r['completion_tokens'],
                    total_tokens=r['total_tokens'],
                )
        return None
    
    def get_summary(self) -> dict:
        total_cost = sum(r['cost_usd'] for r in self.request_history)
        avg_latency = np.mean([r['latency_ms'] for r in self.request_history]) if self.request_history else 0
        avg_tps = np.mean([r['tokens_per_second'] for r in self.request_history]) if self.request_history else 0
        
        return {
            'total_requests': self.total_requests,
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_tokens': self.total_input_tokens + self.total_output_tokens,
            'total_cost_usd': total_cost,
            'avg_latency_ms': avg_latency,
            'avg_tokens_per_second': avg_tps,
        }

# Demo: Token usage tracking
tracker = TokenUsageTracker(input_price_per_1k=0.15, output_price_per_1k=0.60)

# Simulate several requests
simulated_requests = [
    ('req_001', 'llama-3-8b', 150, 45, 1200),
    ('req_002', 'llama-3-8b', 80, 120, 3500),
    ('req_003', 'llama-3-8b', 500, 200, 5000),
    ('req_004', 'llama-3-8b', 30, 15, 450),
    ('req_005', 'llama-3-8b', 200, 100, 2800),
]

print("=== Token Usage Records ===")
for rid, model, inp, out, lat in simulated_requests:
    record = tracker.record_request(rid, model, inp, out, lat)
    print(f"  {rid}: {inp} in + {out} out = {inp+out} total, "
          f"cost=${record['cost_usd']:.4f}, "
          f"{record['tokens_per_second']:.1f} tok/s")

summary = tracker.get_summary()
print(f"\n=== Usage Summary ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## 5. Error Handling

A production API must handle errors gracefully and return informative error responses. The OpenAI API uses standard HTTP status codes with a structured error body.

In [ ]:
class APIError:
    """Standard error response format."""
    
    ERROR_TYPES = {
        400: ('invalid_request_error', 'The request was invalid or malformed.'),
        401: ('authentication_error', 'Invalid API key provided.'),
        403: ('permission_error', 'You do not have permission to access this resource.'),
        404: ('not_found_error', 'The requested resource was not found.'),
        429: ('rate_limit_error', 'Too many requests. Please slow down.'),
        500: ('server_error', 'An internal server error occurred.'),
        503: ('service_unavailable', 'The service is temporarily overloaded.'),
    }
    
    @classmethod
    def create(cls, status_code: int, message: str, param: str = None) -> dict:
        error_type, _ = cls.ERROR_TYPES.get(status_code, ('unknown_error', ''))
        error = {
            'error': {
                'message': message,
                'type': error_type,
                'code': status_code,
            }
        }
        if param:
            error['error']['param'] = param
        return error
    
    @classmethod
    def invalid_request(cls, message: str, param: str = None) -> dict:
        return cls.create(400, message, param)
    
    @classmethod
    def model_not_found(cls, model: str) -> dict:
        return cls.create(404, f"The model '{model}' does not exist.")
    
    @classmethod
    def context_length_exceeded(cls, prompt_tokens: int, max_tokens: int, 
                                  context_limit: int) -> dict:
        return cls.create(
            400,
            f"This model's maximum context length is {context_limit} tokens. "
            f"However, your messages resulted in {prompt_tokens} tokens and you "
            f"requested {max_tokens} max_tokens ({prompt_tokens + max_tokens} total). "
            f"Please reduce your prompt or max_tokens.",
            param='max_tokens'
        )
    
    @classmethod
    def rate_limit(cls, requests_per_minute: int) -> dict:
        return cls.create(
            429,
            f"Rate limit exceeded: {requests_per_minute} requests per minute. "
            f"Please retry after a brief wait."
        )
    
    @classmethod
    def server_overloaded(cls) -> dict:
        return cls.create(
            503,
            "The server is currently overloaded. Please retry your request after a brief wait."
        )

# Demo: Error responses
print("=== API Error Responses ===")
errors = [
    APIError.invalid_request("temperature must be between 0 and 2", param="temperature"),
    APIError.model_not_found("gpt-4-turbo"),
    APIError.context_length_exceeded(7500, 2000, 8192),
    APIError.rate_limit(60),
    APIError.server_overloaded(),
]

for err in errors:
    print(f"\n  Status {err['error']['code']}: {err['error']['type']}")
    print(f"  Message: {err['error']['message'][:80]}...")

## 6. Complete API Server Implementation

Now let's put everything together into a complete OpenAI-compatible API server. We'll use a class-based design that could easily be wrapped with FastAPI.

In [ ]:
import random

class MockLLMEngine:
    """Simulates an LLM inference engine."""
    
    def __init__(self, model_name: str, context_length: int = 8192):
        self.model_name = model_name
        self.context_length = context_length
        self.vocab = [
            "The", " capital", " of", " France", " is", " Paris", ".",
            " It", " has", " been", " the", " political", " and", " cultural",
            " center", " since", " ancient", " times", " known", " for",
            " its", " beautiful", " architecture", " museums", " cuisine",
        ]
    
    def count_tokens(self, text: str) -> int:
        """Approximate token count."""
        return len(text.split()) + len(text) // 10
    
    def generate(self, prompt_tokens: int, max_tokens: int, 
                 params: SamplingParams) -> List[str]:
        """Generate a list of text tokens."""
        num_tokens = min(max_tokens, random.randint(5, max_tokens))
        tokens = []
        for i in range(num_tokens):
            tokens.append(random.choice(self.vocab))
        return tokens


class OpenAICompatibleServer:
    """Complete OpenAI-compatible API server."""
    
    def __init__(self):
        self.engines: Dict[str, MockLLMEngine] = {}
        self.usage_tracker = TokenUsageTracker()
        self.rate_limiter: Dict[str, List[float]] = {}
        self.max_rpm = 60
    
    def register_model(self, model_name: str, context_length: int = 8192):
        self.engines[model_name] = MockLLMEngine(model_name, context_length)
    
    def list_models(self) -> dict:
        """GET /v1/models"""
        return {
            'object': 'list',
            'data': [{
                'id': name,
                'object': 'model',
                'created': int(time.time()),
                'owned_by': 'local',
            } for name in self.engines]
        }
    
    def chat_completions(self, request_body: dict, 
                          api_key: str = 'test') -> dict:
        """POST /v1/chat/completions (non-streaming)"""
        start_time = time.time()
        
        # 1. Parse request
        try:
            request = ChatCompletionRequest.from_dict(request_body)
        except Exception as e:
            return APIError.invalid_request(str(e))
        
        # 2. Validate model
        if request.model not in self.engines:
            return APIError.model_not_found(request.model)
        
        engine = self.engines[request.model]
        
        # 3. Validate parameters
        try:
            params = SamplingParams(request)
        except ValueError as e:
            return APIError.invalid_request(str(e))
        
        # 4. Check rate limit
        now = time.time()
        if api_key not in self.rate_limiter:
            self.rate_limiter[api_key] = []
        self.rate_limiter[api_key] = [t for t in self.rate_limiter[api_key] if now - t < 60]
        if len(self.rate_limiter[api_key]) >= self.max_rpm:
            return APIError.rate_limit(self.max_rpm)
        self.rate_limiter[api_key].append(now)
        
        # 5. Count prompt tokens
        prompt_text = '\n'.join(m.content for m in request.messages)
        prompt_tokens = engine.count_tokens(prompt_text)
        max_tokens = request.max_tokens or (engine.context_length - prompt_tokens)
        
        # 6. Check context length
        if prompt_tokens + max_tokens > engine.context_length:
            return APIError.context_length_exceeded(
                prompt_tokens, max_tokens, engine.context_length
            )
        
        # 7. Generate!
        generated_tokens = engine.generate(prompt_tokens, max_tokens, params)
        completion_text = ''.join(generated_tokens)
        completion_token_count = len(generated_tokens)
        
        # 8. Check stop sequences
        finish_reason = 'stop' if completion_token_count < max_tokens else 'length'
        for stop_seq in params.stop:
            if stop_seq in completion_text:
                completion_text = completion_text[:completion_text.index(stop_seq)]
                finish_reason = 'stop'
                break
        
        # 9. Track usage
        latency_ms = (time.time() - start_time) * 1000
        request_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
        self.usage_tracker.record_request(
            request_id, request.model, prompt_tokens, completion_token_count, latency_ms
        )
        
        # 10. Build response
        response = ChatCompletionResponse(
            id=request_id,
            model=request.model,
            choices=[ChatChoice(
                index=0,
                message=ChatMessage(role='assistant', content=completion_text),
                finish_reason=finish_reason,
            )],
            usage=UsageInfo(
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_token_count,
                total_tokens=prompt_tokens + completion_token_count,
            )
        )
        
        return response.to_dict()
    
    def chat_completions_stream(self, request_body: dict) -> List[str]:
        """POST /v1/chat/completions (streaming) - returns list of SSE events."""
        # Parse and validate (same as above)
        request = ChatCompletionRequest.from_dict(request_body)
        if request.model not in self.engines:
            return [f"data: {json.dumps(APIError.model_not_found(request.model))}\n\n"]
        
        engine = self.engines[request.model]
        params = SamplingParams(request)
        prompt_text = '\n'.join(m.content for m in request.messages)
        prompt_tokens = engine.count_tokens(prompt_text)
        max_tokens = request.max_tokens or 100
        
        # Generate
        generated_tokens = engine.generate(prompt_tokens, max_tokens, params)
        
        request_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
        streamer = StreamingResponseGenerator(request_id, request.model)
        
        events = []
        events.append(streamer.generate_role_chunk())
        for token_text in generated_tokens:
            events.append(streamer.generate_content_chunk(token_text))
        events.append(streamer.generate_final_chunk('stop'))
        events.append(streamer.generate_done())
        
        return events

# Test the complete server
server = OpenAICompatibleServer()
server.register_model('llama-3-8b-instruct', context_length=8192)
server.register_model('llama-3-70b-instruct', context_length=8192)

# List models
print("=== GET /v1/models ===")
print(json.dumps(server.list_models(), indent=2))

# Chat completion (non-streaming)
print("\n=== POST /v1/chat/completions ===")
response = server.chat_completions({
    'model': 'llama-3-8b-instruct',
    'messages': [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'What is the capital of France?'}
    ],
    'temperature': 0.7,
    'max_tokens': 50,
})
print(json.dumps(response, indent=2))

# Error case: model not found
print("\n=== Error: Model Not Found ===")
error_response = server.chat_completions({
    'model': 'gpt-4',
    'messages': [{'role': 'user', 'content': 'hi'}]
})
print(json.dumps(error_response, indent=2))

In [ ]:
# Test streaming
print("=== POST /v1/chat/completions (streaming) ===")
events = server.chat_completions_stream({
    'model': 'llama-3-8b-instruct',
    'messages': [{'role': 'user', 'content': 'Tell me about Paris.'}],
    'max_tokens': 15,
    'stream': True,
})

print("SSE Events:")
for event in events:
    print(f"  {event.strip()}")

## 7. FastAPI Server Template

Here's how you would wrap this into a real FastAPI application. This code is for reference -- it won't run in Jupyter but shows the production pattern.

In [ ]:
# FastAPI server template (for reference)
# In production, this would be in a .py file run with uvicorn

FASTAPI_TEMPLATE = '''
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
from typing import Optional, List, Union
import asyncio
import json
import uvicorn

app = FastAPI(title="LLM Serving API", version="1.0.0")

# Initialize the server (in practice, load real model)
server = OpenAICompatibleServer()
server.register_model('llama-3-8b-instruct')


class ChatRequest(BaseModel):
    model: str
    messages: List[dict]
    temperature: float = Field(default=1.0, ge=0, le=2)
    top_p: float = Field(default=1.0, ge=0, le=1)
    max_tokens: Optional[int] = None
    stream: bool = False
    stop: Optional[Union[str, List[str]]] = None
    n: int = Field(default=1, ge=1)


@app.get("/v1/models")
async def list_models():
    return server.list_models()


@app.post("/v1/chat/completions")
async def chat_completions(request: ChatRequest):
    request_dict = request.dict()
    
    if request.stream:
        # Streaming response
        async def event_generator():
            events = server.chat_completions_stream(request_dict)
            for event in events:
                yield event
                await asyncio.sleep(0.01)  # Simulate token generation delay
        
        return StreamingResponse(
            event_generator(),
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
            }
        )
    else:
        # Non-streaming response
        response = server.chat_completions(request_dict)
        if 'error' in response:
            raise HTTPException(
                status_code=response['error']['code'],
                detail=response['error']
            )
        return JSONResponse(content=response)


@app.get("/health")
async def health_check():
    return {"status": "healthy", "models": list(server.engines.keys())}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("=== FastAPI Server Template ===")
print(FASTAPI_TEMPLATE)
print("\nTo run: uvicorn server:app --host 0.0.0.0 --port 8000")
print("Then use: curl http://localhost:8000/v1/chat/completions -d '{...}'")

## 8. Monitoring and Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt

# Generate more traffic for visualization
np.random.seed(42)
for i in range(50):
    prompt_len = np.random.randint(20, 500)
    max_tok = np.random.randint(10, 200)
    server.chat_completions({
        'model': 'llama-3-8b-instruct',
        'messages': [{'role': 'user', 'content': 'x ' * prompt_len}],
        'max_tokens': max_tok,
    })

# Visualize usage statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

history = server.usage_tracker.request_history

# Plot 1: Token usage over requests
ax = axes[0, 0]
prompt_toks = [r['prompt_tokens'] for r in history]
comp_toks = [r['completion_tokens'] for r in history]
ax.bar(range(len(history)), prompt_toks, label='Prompt tokens', color='#3498db', alpha=0.7)
ax.bar(range(len(history)), comp_toks, bottom=prompt_toks, label='Completion tokens', 
       color='#e74c3c', alpha=0.7)
ax.set_xlabel('Request #')
ax.set_ylabel('Tokens')
ax.set_title('Token Usage per Request', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Latency distribution
ax = axes[0, 1]
latencies = [r['latency_ms'] for r in history]
ax.hist(latencies, bins=20, color='#2ecc71', alpha=0.7, edgecolor='black')
ax.axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean: {np.mean(latencies):.0f}ms')
ax.axvline(np.percentile(latencies, 99), color='orange', linestyle='--', 
           label=f'P99: {np.percentile(latencies, 99):.0f}ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Latency Distribution', fontweight='bold')
ax.legend()

# Plot 3: Throughput (tokens/sec)
ax = axes[1, 0]
tps = [r['tokens_per_second'] for r in history]
ax.plot(tps, 'o-', color='#f39c12', markersize=3, alpha=0.7)
ax.axhline(np.mean(tps), color='red', linestyle='--', label=f'Mean: {np.mean(tps):.0f} tok/s')
ax.set_xlabel('Request #')
ax.set_ylabel('Tokens/Second')
ax.set_title('Generation Throughput', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Cost accumulation
ax = axes[1, 1]
cumulative_cost = np.cumsum([r['cost_usd'] for r in history])
ax.plot(cumulative_cost, '-', color='#9b59b6', linewidth=2)
ax.set_xlabel('Request #')
ax.set_ylabel('Cumulative Cost (USD)')
ax.set_title('Cumulative API Cost', fontweight='bold')
ax.grid(alpha=0.3)
ax.text(len(history) * 0.7, cumulative_cost[-1] * 0.9, 
        f'Total: ${cumulative_cost[-1]:.4f}', fontsize=12, fontweight='bold')

plt.suptitle('API Server Monitoring Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Exercises

### Exercise 1: Add Batch Processing Endpoint

Implement a `/v1/chat/completions/batch` endpoint that accepts multiple requests and processes them efficiently.

In [ ]:
# Exercise 1: Batch processing endpoint

class BatchProcessor:
    """YOUR CODE: Implement batch processing.
    
    Features:
    1. Accept a list of chat completion requests
    2. Process them in parallel (or batched)
    3. Return a list of responses
    4. Track aggregate usage across all requests
    5. Handle individual request failures without failing the entire batch
    """
    
    def __init__(self, server: OpenAICompatibleServer):
        self.server = server
    
    def process_batch(self, requests: List[dict]) -> dict:
        """YOUR CODE: Process a batch of requests.
        
        Returns:
        {
            'responses': [...],  # Individual responses
            'errors': [...],     # Any errors
            'usage': {...},      # Aggregate usage
        }
        """
        pass

print("Exercise 1: Implement BatchProcessor.process_batch()")
print("Test with 5 valid requests and 1 invalid request.")

### Exercise 2: Implement Logprobs

Add `logprobs` support to the API: return the log-probabilities of the top tokens at each generation step.

In [ ]:
# Exercise 2: Logprobs implementation

class LogprobsResult:
    """YOUR CODE: Implement logprobs tracking.
    
    For each generated token, track:
    1. The chosen token and its logprob
    2. Top-N alternative tokens and their logprobs
    
    Return format (matching OpenAI API):
    {
        'content': [
            {
                'token': 'The',
                'logprob': -0.5,
                'top_logprobs': [
                    {'token': 'The', 'logprob': -0.5},
                    {'token': 'A', 'logprob': -1.2},
                    ...
                ]
            },
            ...
        ]
    }
    """
    pass

print("Exercise 2: Implement logprobs support.")
print("Modify the generate() method to return logprobs alongside tokens.")

### Exercise 3: Rate Limiting with Token Buckets

Implement a proper token-bucket rate limiter that limits both requests-per-minute AND tokens-per-minute.

In [ ]:
# Exercise 3: Token bucket rate limiter

class TokenBucketRateLimiter:
    """YOUR CODE: Implement token bucket rate limiting.
    
    Two separate buckets:
    1. Request bucket: max N requests per minute
    2. Token bucket: max M tokens per minute
    
    A request is allowed only if both buckets have capacity.
    Tokens are consumed after generation (not before).
    """
    
    def __init__(self, rpm: int = 60, tpm: int = 100000):
        self.rpm = rpm
        self.tpm = tpm
    
    def check(self, api_key: str) -> tuple:
        """YOUR CODE: Check if request is allowed.
        Returns (allowed: bool, retry_after_ms: float)"""
        pass
    
    def consume(self, api_key: str, tokens: int):
        """YOUR CODE: Consume tokens from the bucket."""
        pass

print("Exercise 3: Implement TokenBucketRateLimiter.")
print("Test with bursts of requests and verify rate limiting works.")

## 10. Summary

In this chapter, we built a complete OpenAI-compatible API layer:

1. **API Schema**: Full request/response models matching the OpenAI specification
2. **Streaming (SSE)**: Real-time token delivery using Server-Sent Events with proper chunked format
3. **Parameter Handling**: Temperature, top_p, penalties, stop sequences, max_tokens with validation
4. **Token Tracking**: Usage counting for billing, monitoring, and rate limiting
5. **Error Handling**: Structured errors with proper HTTP status codes and informative messages
6. **FastAPI Integration**: Production-ready server template with async support
7. **Monitoring**: Dashboard-style metrics visualization

### Key Design Principles

- **Validate early**: Reject bad requests before consuming GPU resources
- **Stream by default**: Streaming improves perceived latency dramatically
- **Track everything**: Token usage, latency, errors -- you can't optimize what you don't measure
- **Fail gracefully**: Individual request failures shouldn't crash the server
- **Be compatible**: OpenAI API compatibility enables the entire ecosystem

## What's Next?

In **Chapter 2.6: Multi-Model Serving & Model Routing**, we'll learn how to serve multiple models from a single server, implementing routing strategies, dynamic model loading, and A/B testing.